In [1]:
import os
from PIL import Image
from transformers import AutoImageProcessor

from videollava.model.builder import load_pretrained_model
from videollava.mm_utils import get_model_name_from_path
from videollava.eval.inference import run_inference_single

TEOCHAT_LOCAL = "/home/gridsan/manderson/.cache/huggingface/hub/models--jirvin16--TEOChat/snapshots/a727ec6baabcaea1bf621d226f42126eda3cc7c2"
LB_LOCAL = "/home/gridsan/manderson/.cache/huggingface/hub/models--LanguageBind--LanguageBind_Image/snapshots/d8c2e37b439f4fc47c649dc8b90cdcd3a4e0c80e"
imagery_path = "/home/gridsan/manderson/skyscraper-s2/data/skyscraper_gdelt_sentinel/imagery/1021434538"

model_name = get_model_name_from_path(TEOCHAT_LOCAL)

tokenizer, model, processor_dict, context_len = load_pretrained_model(
    model_path=TEOCHAT_LOCAL,
    model_base=None,
    model_name=model_name,
    load_8bit=True,
    load_4bit=False,
    device="cuda",
)
model = model.half()

# force local LanguageBind vision tower
vt = model.get_model().get_image_tower()
vt.image_tower_name = LB_LOCAL
vt.load_model()

# processor from same local LanguageBind path
image_processor = AutoImageProcessor.from_pretrained(
    LB_LOCAL,
    trust_remote_code=True
)

image_paths = sorted([
    os.path.join(imagery_path, f)
    for f in os.listdir(imagery_path)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

images = [Image.open(p).convert("RGB") for p in image_paths]

/home/gridsan/manderson/.conda/envs/skyscraper2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/gridsan/manderson/.conda/envs/skyscraper2/lib/python3.11/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/gridsan/manderson/.conda/envs/skyscraper2/lib/python3.11/site-packages/torchvision/transforms/_functional_video.py:6: UserWarning: The 'torchvision.transforms._functional_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms.functional' module instead.
  warnings.warn(
/home/gridsan/manderson/.cond

In [3]:
prompt = "These are satellite images over time: <video>\nDescribe what is happening."

model = model.to("cuda")

out = run_inference_single(
    model=model,
    processor=image_processor,
    tokenizer=tokenizer,
    inp=prompt,
    image_paths=images,   # PIL images
)

print(out)

It looks like there was a wildfire in the area. Initially, the land was green, but it was later cleared.
